In [2]:
import sys
import os

root_path = os.path.abspath(os.path.join('..'))

if root_path not in sys.path:
    sys.path.append(root_path)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.utils import get_year, count_items, owners_midpoint

In [4]:
data = pd.read_csv('..\\data\\processed\\data.csv', index_col=False)
data.head(3)

,appID,name,release_date,required_age,price,dlc_count,reviews,metacritic_score,achievements,recommendations,...,genres,user_score,score_rank,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,peak_ccu,tags
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.00,0,NaN,0,0,0,...,[],0,NaN,0,0,0 - 0,0,0,0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,NaN,0,0,231,...,['Adventure'],0,NaN,252,3,0 - 20000,8,0,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,NaN,0,0,0,...,['Casual'],0,NaN,21,3,0 - 20000,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."


In [4]:
data.shape


(122611, 25)

In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 122611 entries, 0 to 122610
Data columns (total 25 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   appID                     122611 non-null  int64  
 1   name                      122610 non-null  str    
 2   release_date              122611 non-null  str    
 3   required_age              122611 non-null  int64  
 4   price                     122611 non-null  float64
 5   dlc_count                 122611 non-null  int64  
 6   reviews                   12070 non-null   str    
 7   metacritic_score          122611 non-null  int64  
 8   achievements              122611 non-null  int64  
 9   recommendations           122611 non-null  int64  
 10  supported_languages       122611 non-null  str    
 11  full_audio_languages      122611 non-null  str    
 12  developers                122611 non-null  str    
 13  publishers                122611 non-null  str    
 14 

In [6]:
data.isna().mean().sort_values(ascending=False)

score_rank                  0.999674
reviews                     0.901559
name                        0.000008
required_age                0.000000
appID                       0.000000
price                       0.000000
dlc_count                   0.000000
metacritic_score            0.000000
release_date                0.000000
achievements                0.000000
recommendations             0.000000
full_audio_languages        0.000000
supported_languages         0.000000
publishers                  0.000000
categories                  0.000000
genres                      0.000000
developers                  0.000000
user_score                  0.000000
positive                    0.000000
negative                    0.000000
estimated_owners            0.000000
average_playtime_forever    0.000000
average_playtime_2weeks     0.000000
peak_ccu                    0.000000
tags                        0.000000
dtype: float64

In [7]:
data['score_rank'].isna().value_counts()



score_rank
True     122571
False        40
Name: count, dtype: int64

In [8]:
data['estimated_owners'].value_counts().head(20)

estimated_owners
0 - 20000                75404
0 - 0                    21641
20000 - 50000            11396
50000 - 100000            5355
100000 - 200000           3454
200000 - 500000           2853
500000 - 1000000          1154
1000000 - 2000000          729
2000000 - 5000000          405
5000000 - 10000000         125
10000000 - 20000000         51
20000000 - 50000000         31
50000000 - 100000000         9
100000000 - 200000000        4
Name: count, dtype: int64

In [9]:
data['reviews'].isna().value_counts()

reviews
True     110541
False     12070
Name: count, dtype: int64

In [5]:
data = data.drop(columns=['score_rank', 'reviews'])
data

,appID,name,release_date,required_age,price,dlc_count,metacritic_score,achievements,recommendations,supported_languages,...,categories,genres,user_score,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,peak_ccu,tags
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.00,0,0,0,0,[],...,[],[],0,0,0,0 - 0,0,0,0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,0,0,231,['English'],...,"['Single-player', 'Steam Trading Cards', 'Stea...",['Adventure'],0,252,3,0 - 20000,8,0,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,0,0,0,"['English', 'French', 'German', 'Russian']",...,"['Single-player', 'Family Sharing']",['Casual'],0,21,3,0 - 20000,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,0,19,0,['Korean'],...,"['Single-player', 'Steam Achievements', 'Famil...","['Casual', 'Indie', 'Simulation']",0,0,0,0 - 20000,0,0,1,[]
4,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,0,0,0,['English'],...,"['Single-player', 'VR Only', 'Steam Leaderboar...","['Action', 'Early Access']",0,0,0,0 - 20000,0,0,0,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122606,4152910,完美传奇,"Jan 4, 2026",0,0.00,0,0,0,0,"['Simplified Chinese', 'Traditional Chinese']",...,"['Single-player', 'Multi-player', 'MMO', 'PvP'...","['Action', 'Adventure', 'Massively Multiplayer...",0,0,0,0 - 0,0,0,0,[]
122607,4042800,Poker Fate - ACG Texas Hold'em,"Jan 3, 2026",0,0.00,0,0,0,0,"['English', 'Japanese', 'Simplified Chinese', ...",...,"['Multi-player', 'PvP', 'Online PvP', 'Cross-P...","['Casual', 'Strategy', 'Free To Play']",0,0,0,0 - 0,0,0,0,[]
122608,3522550,Adira Nusantara,"Jan 3, 2026",0,7.99,0,0,0,0,['English'],...,"['Single-player', 'Custom Volume Controls', 'N...","['Action', 'Adventure', 'Casual', 'Early Access']",0,0,0,0 - 0,0,0,0,[]
122609,3680350,A Lenda de Niterói,"Jan 4, 2026",0,2.09,0,0,0,0,"['Portuguese - Brazil', 'English']",...,"['Single-player', 'Full controller support', '...","['Action', 'Adventure', 'Casual', 'Indie']",0,0,0,0 - 0,0,0,0,[]


#Variables derivadas

In [ ]:
""" 
review_count
review_ratio

num_languages
num_audio_languages

release_year

owners_midpoint
"""

data['review_count'] = data['positive'] + data['negative']
data['review_radio'] = data['positive'] / data['review_count']
data['release_year'] = (data['release_date']).apply(get_year)
data['num_languages'] = (data['supported_languages']).apply(count_items)
data['num_audio_languages'] = (data['full_audio_languages']).apply(count_items)
data['review_score_prc'] = 100 * data['review_radio']
data['owners_midpoint'] = data['estimated_owners'].apply(owners_midpoint)
data['log_owners'] = np.log1p(data['owners_midpoint'])
data.head()

,appID,name,release_date,required_age,price,dlc_count,metacritic_score,achievements,recommendations,supported_languages,...,peak_ccu,tags,review_count,review_radio,release_year,num_languages,num_audio_languages,review_score_prc,owners_midpoint,log_owners
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.00,0,0,0,0,[],...,0,[],0,NaN,2023,0,0,NaN,0.0,0.00000
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,0,0,231,['English'],...,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':...",255,0.988235,2016,1,0,98.823529,10000.0,9.21044
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,0,0,0,"['English', 'French', 'German', 'Russian']",...,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4...",24,0.875000,2019,4,0,87.500000,10000.0,9.21044
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,0,19,0,['Korean'],...,1,[],0,NaN,2024,1,1,NaN,10000.0,9.21044
4,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,0,0,0,['English'],...,0,[],0,NaN,2025,1,1,NaN,10000.0,9.21044
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122606,4152910,完美传奇,"Jan 4, 2026",0,0.00,0,0,0,0,"['Simplified Chinese', 'Traditional Chinese']",...,0,[],0,NaN,2026,2,2,NaN,0.0,0.00000
122607,4042800,Poker Fate - ACG Texas Hold'em,"Jan 3, 2026",0,0.00,0,0,0,0,"['English', 'Japanese', 'Simplified Chinese', ...",...,0,[],0,NaN,2026,4,2,NaN,0.0,0.00000
122608,3522550,Adira Nusantara,"Jan 3, 2026",0,7.99,0,0,0,0,['English'],...,0,[],0,NaN,2026,1,1,NaN,0.0,0.00000
122609,3680350,A Lenda de Niterói,"Jan 4, 2026",0,2.09,0,0,0,0,"['Portuguese - Brazil', 'English']",...,0,[],0,NaN,2026,2,2,NaN,0.0,0.00000


In [10]:
print(data['release_year'].dtype)

int64


In [12]:
data['game_age'] = 2026 - data['release_year']
data.head()

,appID,name,release_date,required_age,price,dlc_count,metacritic_score,achievements,recommendations,supported_languages,...,tags,review_count,review_radio,release_year,num_languages,num_audio_languages,review_score_prc,owners_midpoint,log_owners,game_age
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.00,0,0,0,0,[],...,[],0,NaN,2023,0,0,NaN,0.0,0.00000,3
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,0,0,231,['English'],...,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':...",255,0.988235,2016,1,0,98.823529,10000.0,9.21044,10
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,0,0,0,"['English', 'French', 'German', 'Russian']",...,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4...",24,0.875000,2019,4,0,87.500000,10000.0,9.21044,7
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,0,19,0,['Korean'],...,[],0,NaN,2024,1,1,NaN,10000.0,9.21044,2
4,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,0,0,0,['English'],...,[],0,NaN,2025,1,1,NaN,10000.0,9.21044,1


In [13]:
data.isna().mean().sort_values(ascending=False)

review_radio                0.323478
review_score_prc            0.323478
name                        0.000008
appID                       0.000000
price                       0.000000
dlc_count                   0.000000
release_date                0.000000
required_age                0.000000
recommendations             0.000000
supported_languages         0.000000
full_audio_languages        0.000000
developers                  0.000000
publishers                  0.000000
categories                  0.000000
metacritic_score            0.000000
achievements                0.000000
user_score                  0.000000
genres                      0.000000
positive                    0.000000
negative                    0.000000
average_playtime_2weeks     0.000000
peak_ccu                    0.000000
estimated_owners            0.000000
average_playtime_forever    0.000000
review_count                0.000000
tags                        0.000000
num_languages               0.000000
r

In [14]:
data.to_csv(
    '../data/processed/steam_games_clean.csv',
    index=False
)